# Prática guiada: previsão de idade de abalones

Um único modelo com **duas cabeças (dual head)**: uma prevê a idade como
número contínuo (regressão) e a outra classifica o animal em jovem, adulto
ou idoso (classificação).

Dataset: [Abalone — UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/1/abalone)

## [1] Importação de bibliotecas

In [ ]:
"""
Define uma seed global para garantir reprodutibilidade máxima
no treinamento com TensorFlow/Keras.

NOTE 1: Para aulas, deixo essa parte comentada para demonstrar o
        comportamento estocástico das práticas com DL.
NOTE 2: Na prática diária com DL, determinamos as SEEDs somente no 
        split dos dados, pois operações em GPU sofrem perda de 
        performance quando forçadas ao determinismo.
NOTE 3: Em alguns casos de treinamento avançado com DL, não é 
        possível estabelecer 100% de determinismo devido à alta
        dependência de operações paralelizadas.
"""
# 1. Variáveis de ambiente (precisam ser definidas ANTES de importar TF!
import os
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'      # força kernels determinísticos
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'    # determinismo em GPU (cuDNN)


import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, RobustScaler, OneHotEncoder
from sklearn.utils import compute_class_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from absl import logging

# seeds determinadas após importação dos módulos
keras.utils.set_random_seed(SEED)
# 3. Habilita operações determinísticas no TF (TF >= 2.8)
# NOTE: Haverá perda de performance
# NOTE: Em alguns casos, ainda não será 100% determinístico (paralelismo!)
tf.config.experimental.enable_op_determinism()

# silenciar alguns avisos do tf que não serão relevantes
tf.get_logger().setLevel('ERROR')
logging.set_verbosity(logging.ERROR)

## [2] Modelos usados nesta prática

### `compile_model` - perdas, pesos e métricas de cada head

Cada head tem sua própria loss e suas próprias métricas. `loss_weights`
equilibra as duas: a perda de regressão (MSE, na casa das dezenas) é
dividida por 100 para não dominar a de classificação.

In [ ]:
def compile_model(model, lr=1e-3):
    reg_metrics = [
        keras.metrics.MeanAbsoluteError(name="mae"),
        keras.metrics.R2Score(name="R2")
    ]
    cat_metrics = [
        keras.metrics.CategoricalAccuracy(name="acc"),
        keras.metrics.F1Score(name="F1", average="weighted")
    ]

    model.compile(
        optimizer=keras.optimizers.Adam(lr,),
        loss={
            "reg_output": keras.losses.MeanSquaredError(),
            "cat_output": keras.losses.CategoricalCrossentropy()
        },
        loss_weights={
            "reg_output": 1/100,
            "cat_output": 1
        },
        metrics={
            "reg_output": reg_metrics,
            "cat_output": cat_metrics
        }
    )
    return model

### Arquiteturas `dual_head_mlp_*`

Quatro variações do mesmo esqueleto, camadas densas compartilhadas que
se dividem em duas saídas; da mais simples (`_1`) à mais profunda e
regularizada (`_4`).

In [ ]:
def dual_head_mlp_1(input_shape):
    inputs = keras.Input(input_shape, name='features')

    # hidden layers compartilhados
    x = keras.layers.Dense(16, activation='relu')(inputs)
    x = keras.layers.Dense(8, activation='relu')(x)

    # head de regressão
    reg_output = keras.layers.Dense(1, name='reg_output')(x)

    # head de classificação
    cat_output = keras.layers.Dense(3, activation='softmax', name='cat_output')(x)

    model = keras.Model(inputs, outputs={'reg_output': reg_output, 'cat_output': cat_output})
    return model


def dual_head_mlp_2(input_shape):
    inputs = keras.Input(input_shape, name='features')

    # hidden layers compartilhados
    x = keras.layers.Dense(64, activation='relu')(inputs)
    x = keras.layers.Dense(32, activation='relu')(x)
    x = keras.layers.Dense(16, activation='relu')(x)

    # head de regressão
    reg_output = keras.layers.Dense(1, name='reg_output')(x)

    # head de classificação
    cat_output = keras.layers.Dense(3, activation='softmax', name='cat_output')(x)

    model = keras.Model(inputs, outputs={'reg_output': reg_output, 'cat_output': cat_output})
    return model


def dual_head_mlp_3(input_shape):
    inputs = keras.Input(input_shape, name='features')

    # hidden layers compartilhados
    x = keras.layers.Dense(64, activation='relu')(inputs)
    x = keras.layers.Dropout(0.25)(x)
    x = keras.layers.Dense(32, activation='relu')(x)
    x = keras.layers.Dropout(0.25)(x)
    x = keras.layers.Dense(16, activation='relu')(x)
    x = keras.layers.Dropout(0.4)(x)

    # head de regressão
    reg_output = keras.layers.Dense(1, name='reg_output')(x)

    # head de classificação
    cat_output = keras.layers.Dense(3, activation='softmax', name='cat_output')(x)

    model = keras.Model(inputs, outputs={'reg_output': reg_output, 'cat_output': cat_output})
    return model


def dual_head_mlp_4(input_shape):
    inputs = keras.Input(input_shape, name='features')

    # hidden layers compartilhados
    x = keras.layers.Dense(256, activation='relu')(inputs)
    x = keras.layers.Dropout(0.4)(x)
    x = keras.layers.Dense(128, activation='relu')(x)
    x = keras.layers.Dropout(0.4)(x)
    x = keras.layers.Dense(64, activation='relu')(x)
    x = keras.layers.Dropout(0.5)(x)

    # head de regressão
    reg_output = keras.layers.Dense(1, name='reg_output')(x)

    # head de classificação
    cat_output = keras.layers.Dense(3, activation='softmax', name='cat_output')(x)

    model = keras.Model(inputs, outputs={'reg_output': reg_output, 'cat_output': cat_output})
    return model

## [3] Funções utilitárias

As células desta seção apenas **definem** as funções usadas no restante do
notebook. Execute todas uma vez, no início da sessão.

### `_coerce_plot_data`: normaliza a entrada dos gráficos

Converte DataFrame, Series ou array NumPy num formato único, para que as
duas funções de grade abaixo aceitem qualquer um deles.

*não se preocupe com os detalhes dessa função, é apenas utilitária*

In [1]:
def _coerce_plot_data(data, column_names=None):
    if isinstance(data, pd.Series):
        plot_data = data.to_frame(name=data.name or 'value')
    elif isinstance(data, pd.DataFrame):
        plot_data = data.copy()
    elif isinstance(data, np.ndarray):
        array = np.asarray(data)
        if array.ndim == 1:
            array = array.reshape(-1, 1)
        elif array.ndim != 2:
            raise ValueError('NumPy array data must be 1D or 2D.')

        if column_names is None:
            if array.shape[1] == 1:
                column_names = ['value']
            else:
                column_names = [f'feature_{index}' for index in range(array.shape[1])]
        elif len(column_names) != array.shape[1]:
            raise ValueError('column_names length must match the number of plotted columns.')

        plot_data = pd.DataFrame(array, columns=list(column_names))
    else:
        raise TypeError('data must be a pandas DataFrame, Series, or NumPy array.')

    if column_names is not None and not isinstance(data, np.ndarray):
        if len(column_names) != len(plot_data.columns):
            raise ValueError('column_names length must match the number of plotted columns.')
        plot_data = plot_data.copy()
        plot_data.columns = list(column_names)

    if plot_data.empty:
        raise ValueError('data must contain at least one column to plot.')

    return plot_data

### `plot_histogram_grid`: grade de histogramas por coluna

*não se preocupe com os detalhes dessa função, é apenas utilitária*

In [2]:
def plot_histogram_grid(
    data, # (N, F)
    bins=20,
    figsize_scale=4.0,
    color='#1f77b4',
    edgecolor='white',
    title=None,
    column_names=None,
):
    """
    Plot distributions for all columns in a pandas DataFrame, Series, or NumPy array.

    Numeric columns are shown as histograms. Categorical, object, and boolean
    columns are shown as category-count bar charts.

    Args:
        data (pd.DataFrame | pd.Series | np.ndarray): Data to visualize.
        bins (int): Number of bins used in each histogram.
        figsize_scale (float): Base size multiplier for each subplot.
        color (str): Fill color for the histogram bars.
        edgecolor (str): Edge color for the histogram bars.
        title (str | None): Optional figure title.
        column_names (Sequence[str] | None): Optional names for plotted columns.
            Useful when data is a NumPy array.

    Returns:
        tuple: The matplotlib figure and axes array.
    """
    plot_data = _coerce_plot_data(data, column_names=column_names)

    num_plots = len(plot_data.columns)
    ncols = int(np.ceil(np.sqrt(num_plots)))
    nrows = int(np.ceil(num_plots / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * figsize_scale, nrows * figsize_scale),
        squeeze=False,
        constrained_layout=True,
    )
    fig.patch.set_facecolor('white')

    axes_flat = axes.ravel()
    for ax, column in zip(axes_flat, plot_data.columns):
        values = plot_data[column].dropna()
        dtype = plot_data[column].dtype
        is_numeric = (
            pd.api.types.is_numeric_dtype(dtype)
            and not pd.api.types.is_bool_dtype(dtype)
            and not isinstance(dtype, pd.CategoricalDtype)
        )

        if values.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes, fontsize=11)
            ax.set_axis_off()
            continue

        if is_numeric:
            ax.hist(values, bins=bins, color=color, edgecolor=edgecolor, linewidth=0.8, alpha=0.9)
        else:
            counts = plot_data[column].value_counts(dropna=False, sort=False)
            labels = ['NaN' if pd.isna(label) else str(label) for label in counts.index]
            ax.bar(labels, counts.values, color=color, edgecolor=edgecolor, linewidth=0.8, alpha=0.9)
            ax.tick_params(axis='x', rotation=30)

        ax.set_title(str(column).replace('_', ' '), fontsize=12, pad=10)
        ax.set_ylabel('Count', fontsize=10)
        ax.set_facecolor('#fafafa')
        ax.grid(axis='y', alpha=0.18)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_alpha(0.25)
        ax.spines['bottom'].set_alpha(0.25)
        ax.tick_params(labelsize=9)

    for ax in axes_flat[num_plots:]:
        ax.set_visible(False)

    fig.suptitle(title or 'Histograms by Column', fontsize=14, fontweight='bold')
    plt.show()
    return fig, axes

### `plot_boxplot_grid`: grade de boxplots por coluna

*não se preocupe com os detalhes dessa função, é apenas utilitária*

In [3]:
def plot_boxplot_grid(
    data,
    figsize_scale=4.0,
    color='#1f77b4',
    title=None,
    column_names=None,
):
    """
    Plot a square grid of boxplots for numeric columns in the input data.

    Args:
        data (pd.DataFrame | pd.Series | np.ndarray): Data to visualize.
        figsize_scale (float): Base size multiplier for each subplot.
        color (str): Main color used for boxplot styling.
        title (str | None): Optional figure title.
        column_names (Sequence[str] | None): Optional names for plotted columns.
            Useful when data is a NumPy array.

    Returns:
        tuple: The matplotlib figure and axes array.
    """
    plot_data = _coerce_plot_data(data, column_names=column_names)
    numeric_data = plot_data.select_dtypes(include=[np.number])
    skipped_columns = [col for col in plot_data.columns if col not in numeric_data.columns]

    if skipped_columns:
        print(f"Skipping non-numeric columns: {', '.join(map(str, skipped_columns))}")

    if numeric_data.empty:
        raise ValueError('data must contain at least one numeric column to plot boxplots.')

    num_plots = len(numeric_data.columns)
    ncols = int(np.ceil(np.sqrt(num_plots)))
    nrows = int(np.ceil(num_plots / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * figsize_scale, nrows * figsize_scale),
        squeeze=False,
        constrained_layout=True,
    )
    fig.patch.set_facecolor('white')

    axes_flat = axes.ravel()
    for ax, column in zip(axes_flat, numeric_data.columns):
        values = numeric_data[column].dropna()

        if values.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes, fontsize=11)
            ax.set_axis_off()
            continue

        ax.boxplot(
            values,
            patch_artist=True,
            widths=0.45,
            boxprops={'facecolor': color, 'alpha': 0.35, 'edgecolor': color, 'linewidth': 1.2},
            medianprops={'color': '#d62728', 'linewidth': 2},
            whiskerprops={'color': color, 'linewidth': 1.2},
            capprops={'color': color, 'linewidth': 1.2},
            flierprops={
                'marker': 'o',
                'markersize': 4,
                'markerfacecolor': '#ff7f0e',
                'markeredgecolor': 'white',
                'alpha': 0.85,
            },
        )
        ax.set_title(str(column).replace('_', ' '), fontsize=12, pad=10)
        ax.set_ylabel('Value', fontsize=10)
        ax.set_xticks([])
        ax.set_facecolor('#fafafa')
        ax.grid(axis='y', alpha=0.18)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_alpha(0.25)
        ax.spines['bottom'].set_alpha(0.25)
        ax.tick_params(labelsize=9)

    for ax in axes_flat[num_plots:]:
        ax.set_visible(False)

    fig.suptitle(title or 'Boxplots by Column', fontsize=14, fontweight='bold')
    plt.show()
    return fig, axes

### `smooth_dataframe`: suavização das séries do histórico

In [4]:
def smooth_dataframe(df, window_size=10):
    smoothed_data = {
        col: np.convolve(df[col], np.ones(window_size) / window_size, mode='valid')
        for col in df.columns
    }
    return pd.DataFrame(smoothed_data)

### `plot_history`: curvas de treino e validação por métrica

*não se preocupe com os detalhes dessa função, é apenas utilitária*

In [ ]:
def plot_history(history_df, skip_epochs=0, smooth=1):
    """
    Plots training and (optional) validation metrics from a Keras history DataFrame.

    Args:
        history_df (pd.DataFrame): DataFrame containing Keras training history.
                                   Expected to have columns like 'loss', 'accuracy', etc.
                                   Validation columns (if present) must be named 'val_<metric>'.
        skip_epochs (int): Number of initial rows (epochs) to skip before plotting.
        smooth (int): Window length for smoothing. If > 1, the DataFrame will be smoothed first.
                      (Assumes you have a function `smooth_dataframe(df, window)` defined elsewhere.)
    """
    if not isinstance(history_df, pd.DataFrame):
        raise TypeError("history_df must be a pandas DataFrame.")
    if history_df.empty:
        print("History DataFrame is empty. Nothing to plot.")
        return

    # Optionally skip some initial epochs
    history_df = history_df.iloc[skip_epochs:].copy()

    # If the user requested smoothing, apply it here.
    if smooth > 1:
        history_df = smooth_dataframe(history_df, smooth)

    # Identify all "training" columns (i.e. those that do NOT start with 'val_').
    # For each such column, check whether a corresponding 'val_<col>' exists.
    base_metric_names = []
    metric_has_val = {}  # dict: metric_name -> True/False
    for col in history_df.columns:
        if not col.startswith('val_'):
            base_metric_names.append(col)
            if f"val_{col}" in history_df.columns:
                metric_has_val[col] = True
            else:
                metric_has_val[col] = False
                print(f"Warning: Found training metric '{col}' but no 'val_{col}'. Will plot training-only.")

    if not base_metric_names:
        print("No training metrics found to plot.")
        return

    num_metrics = len(base_metric_names)

    # Arrange subplots: até 2 colunas, nrows = ceil(num_metrics/ncols)
    ncols = min(2, num_metrics)
    nrows = math.ceil(num_metrics / ncols)
    # squeeze=False garante sempre uma matriz 2D de Axes, mesmo com 1 métrica
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6, nrows * 5), squeeze=False)
    axes_flat = axes.flatten()

    for i, metric_name in enumerate(base_metric_names):
        ax = axes_flat[i]
        train_col = metric_name
        val_col = f"val_{metric_name}"
        
        # Plot training curve
        ax.plot(
            history_df[train_col],
            label=f"Training {metric_name.replace('_', ' ').title()}",
            # marker="o",
            linestyle="-"
        )

        # If a validation column exists, plot it too
        if metric_has_val[metric_name]:
            ax.plot(
                history_df[val_col],
                label=f"Validation {metric_name.replace('_', ' ').title()}",
                # marker="x",
                linestyle="--"
            )

        ax.set_title(f"{metric_name.replace('_', ' ').title()} Over Epochs")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(metric_name.replace("_", " ").title())
        ax.legend()
        ax.grid(True, linestyle="--", alpha=0.7)

        # Compute y-limits using whichever columns are available
        train_min = history_df[train_col].min()
        train_max = history_df[train_col].max()
        if metric_has_val[metric_name]:
            val_min = history_df[val_col].min()
            val_max = history_df[val_col].max()
            overall_min = min(train_min, val_min)
            overall_max = max(train_max, val_max)
        else:
            overall_min = train_min
            overall_max = train_max

        if pd.notna(overall_min) and pd.notna(overall_max) and (overall_max > overall_min):
            padding = (overall_max - overall_min) * 0.1
            ax.set_ylim(overall_min - padding, overall_max + padding)

    # If there are any unused subplot slots, delete them
    for j in range(num_metrics, nrows * ncols):
        fig.delaxes(axes_flat[j])

    plt.tight_layout()
    plt.show()

## [4] Dataset

Os dados vêm direto do repositório da UCI. O arquivo não tem cabeçalho,
então passamos os nomes das colunas a partir da documentação da fonte.

In [ ]:
csv_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"
column_names = ["Sex", "Length", "Diameter", "Height",
                "WholeWeight", "ShuckedWeight",
                "VisceraWeight", "ShellWeight", "Rings"]
data = pd.read_csv(csv_url, names=column_names)

# 'data' guarda o original; todo o tratamento acontece em 'data_aux'
data_aux = data.copy()

data.head()

## [5] Inspeção inicial: procurando anomalias

Os boxplots revelam valores impossíveis em `Height`: alturas iguais a zero
e alturas muito acima do resto da distribuição.

In [ ]:
plot_boxplot_grid(data_aux)

In [ ]:
data_aux.describe()

Alturas absurdamente grandes:

In [ ]:
data_aux[data_aux["Height"] >= 0.5]

Alturas iguais a zero — fisicamente impossíveis:

In [ ]:
data_aux[data_aux["Height"] == 0.]

## [6] Remoção das anomalias de "Height"

In [ ]:
filter_1 = data_aux["Height"] == 0
filter_2 = data_aux["Height"] >= 0.6

mask = ~(filter_1 | filter_2)
data_aux = data_aux[mask].copy()

print(f'linhas removidas: {(~mask).sum()}')
print(f'linhas restantes: {mask.sum()}')

## [7] Heatmap de correlação entre as features numéricas

Todas as medidas físicas são fortemente correlacionadas entre si. Os
abalones simplesmente crescem em todas as dimensões ao mesmo tempo.

In [ ]:
corr_matrix = data_aux.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,          # Mostra os valores numéricos dentro dos quadrados
    cmap='coolwarm',     # Paleta de cores (azul para negativa, vermelho para positiva)
    fmt=".2f",           # Limita o número de casas decimais
    vmin=-1, vmax=1      # Fixa os limites da barra de cores para o padrão da correlação
)
plt.show()

## [8] Estatísticas descritivas: antes e depois da filtragem

Antes:

In [ ]:
data.describe()

Depois:

In [ ]:
data_aux.describe()

## [9] Distribuições: histogramas e boxplots

In [ ]:
plot_histogram_grid(data_aux, bins=20)

In [ ]:
plot_boxplot_grid(data_aux)

## [10] Categorical encoding

One-hot encoding para a variável `Sex`.

Cada categoria recebe um vetor binário de N dimensões, onde N é o número
de classes:

- shape = (N,)
- não captura ordem!

In [ ]:
pd.get_dummies(data_aux, columns=["Sex"], dtype=int).head()

## [11] Target encoding

A idade do abalone não está no dataset: ela é derivada do número de anéis
da concha, pela fórmula `age = Rings + 1.5`.

In [ ]:
data_aux["age"] = data_aux["Rings"] + 1.5

data_aux.head()

Distribuição do target:

In [ ]:
plot_histogram_grid(data_aux["age"], bins=10)

## [12] Discretização de "age" em classes

Abalones são usualmente classificados em jovem, adulto ou idoso. Cortamos
a idade contínua em 3 faixas para criar o target da head de classificação.

**OBS:** Discretizar a idade ou não é apenas opcional. Vai depender exclusivamente do seu problema de negócio. 
Nesta prática, vamos trabalhar com ambos targets simultaneamente para simular um modelo multi-task, que poderia
ser igualmente utilizado para resolver duas tarefas complexas e distintas a partir dos mesmo dados.

In [ ]:
age_bins = [0, 8, 12, data_aux['age'].max()]
age_labels = [0, 1, 2]  # 0 = jovem | 1 = adulto | 2 = idoso
class_names = ['Jovem', 'Adulto', 'Idoso']

data_aux['age_cat'] = pd.cut(data_aux['age'], bins=age_bins, labels=age_labels)

data_aux.head()

As classes ficam desbalanceadas, mas isso será corrigido adiante com
pesos por amostra (seção [16]).

In [ ]:
plot_histogram_grid(data_aux['age_cat'])

## [13] X e y

`Rings`, `age` e `age_cat` saem das features: as três carregam a resposta
e deixá-las na entrada seria vazamento de dados.

In [ ]:
X = data_aux.drop(columns=["Rings", "age", "age_cat"])

y_reg = data_aux["age"].values.astype('float32')

# Target categórico como sparse ou ohe:
# (à seu critério, mas escolha a loss correspondente)
# y_cat = data_aux['age_cat'].values.astype('uint8')              # sparse
y_cat = pd.get_dummies(data_aux["age_cat"], dtype="uint8").values # ohe

print(f'X: {X.shape} | y_reg: {y_reg.shape} | y_cat: {y_cat.shape}')

## [14] Split treino / validação / teste

Dois splits encadeados, ambos estratificados por `y_cat` para preservar a
proporção das três classes em cada conjunto.

In [ ]:
# split 80/20 estratificado de acordo com y_cat (TESTE)
X_train, X_test, y_reg_train, y_reg_test, y_cat_train, y_cat_test = train_test_split(
    X, y_reg, y_cat, test_size=0.2, random_state=42, stratify=y_cat
)

# (VALIDAÇÃO)
X_train, X_val, y_reg_train, y_reg_val, y_cat_train, y_cat_val = train_test_split(
    X_train, y_reg_train, y_cat_train, test_size=0.1, random_state=42, stratify=y_cat_train
)

print(f'train size: {X_train.shape[0]}')
print(f'val size: {X_val.shape[0]}')
print(f'test size: {X_test.shape[0]}')

## [15] Pipeline de pré-processamento (ColumnTransformer)

Cada grupo de colunas recebe o tratamento que lhe cabe:

- **medidas lineares** → `RobustScaler` (robusto a outliers);
- **pesos** → `box-cox` para simetrizar a distribuição, depois `RobustScaler`;
- **`Sex`** → one-hot encoding.

O `fit` acontece **só no treino**; validação e teste apenas usam a
transformação já ajustada.

In [ ]:
rbt_scaler = RobustScaler()
pow_scaler = PowerTransformer(method='box-cox', standardize=False)  # x -> box-cox -> ~std~ -> x_scaled
ohe        = OneHotEncoder(sparse_output=False)

pow_rbt_pipe = Pipeline(steps=[
    ("power", pow_scaler),
    ("rbt",   RobustScaler())
])

# liste os grupos de coluna por tipo de scaler
rbt_cols     = ['Length', 'Diameter', 'Height']
pow_rbt_cols = ['WholeWeight', 'ShuckedWeight', 'VisceraWeight', 'ShellWeight']
cat_cols     = ['Sex']

# lista de operações por features
# (nome, operação, features)
transformers = [
    ("rbt",     rbt_scaler,   rbt_cols),
    ("pow_rbt", pow_rbt_pipe, pow_rbt_cols),
    ("cat",     ohe,          cat_cols)
]

preprocessor = ColumnTransformer(transformers)

X_train_scaled = preprocessor.fit_transform(X_train).astype("float32")
X_val_scaled   = preprocessor.transform(X_val).astype("float32")
X_test_scaled  = preprocessor.transform(X_test).astype("float32")

print(f'X_train_scaled: {X_train_scaled.shape}')

Depois do pipeline, as distribuições ficam centradas e em escalas
comparáveis (as três últimas colunas são o one-hot de `Sex`):

In [ ]:
plot_histogram_grid(X_train_scaled)

## [16] Sample weights

As classes de idade são desbalanceadas. Damos a cada amostra um peso
inversamente proporcional à frequência da sua classe, para que o erro nas
classes raras pese tanto quanto o das comuns.

In [ ]:
y_cat_train_labels = y_cat_train.argmax(axis=-1)

# crie um dicionário de pesos de cada classe {0: w0, 1: w1, ...}
class_weight = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_cat_train_labels),
    y=y_cat_train_labels
)
class_weight = dict(enumerate(class_weight))

# atribua os respectivos pesos a cada uma das amostras
sample_weight = np.array([class_weight[label] for label in y_cat_train_labels])

# pesos por head: só a classificação é reponderada
sample_weight_dict = {
    'reg_output': np.ones_like(y_reg_train),
    'cat_output': sample_weight
}

print({name: round(w, 3) for name, w in zip(class_names, class_weight.values())})

## [17] Modelagem: o modelo dual head

Um tronco de camadas densas compartilhado se divide em duas saídas:
`reg_output` (1 neurônio, sem ativação, um número) e `cat_output`
(3 neurônios com softmax, uma distribuição de probabilidade).

In [ ]:
model = dual_head_mlp_1(X_train_scaled.shape[1:])
model.summary()

In [ ]:
keras.utils.plot_model(model, show_shapes=True, show_layer_names=True)

### Aparte: o que a `softmax` faz

Ela transforma um vetor de números quaisquer em probabilidades: todas
positivas e somando 1. É o que permite ler a saída da head de
classificação como "chance de pertencer a cada classe".

In [ ]:
def softmax(array):
    return np.exp(array) / np.exp(array).sum()


z = [-3, 0, 3]
softmax(z)

## [18] Treino

`EarlyStopping` interrompe o treino quando a `val_loss` para de melhorar e
restaura os melhores pesos, por isso podemos pedir 1000 épocas sem medo.

In [ ]:
model = dual_head_mlp_1(X_train_scaled.shape[1:])
model = compile_model(model, lr=1e-3)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
    ),
]

# Train the model
history = model.fit(
    X_train_scaled,
    {"reg_output": y_reg_train, "cat_output": y_cat_train},
    validation_data=(
        X_val_scaled,
        {"reg_output": y_reg_val, "cat_output": y_cat_val},
    ),
    epochs=1000,
    batch_size=128,
    shuffle=True,
    sample_weight=sample_weight_dict,
    callbacks=callbacks
)
history = pd.DataFrame(history.history)

print(f'épocas treinadas: {len(history)}')

In [ ]:
plot_history(history, skip_epochs=10, smooth=10)

## [19] Avaliação

Comparar treino e teste é o que revela overfit: se as métricas de teste
forem muito piores, o modelo decorou em vez de generalizar.

Desempenho no **treino**:

In [ ]:
model.evaluate(
    X_train_scaled,
    {"reg_output": y_reg_train, "cat_output": y_cat_train},
    return_dict=True
)

Desempenho no **teste**:

In [ ]:
model.evaluate(
    X_test_scaled,
    {"reg_output": y_reg_test, "cat_output": y_cat_test},
    return_dict=True
)

A matriz de confusão mostra *onde* o modelo erra, tipicamente entre
classes vizinhas (adulto/idoso), raramente entre jovem e idoso.

In [ ]:
y_pred = model.predict(X_test_scaled)['cat_output']

cm = confusion_matrix(y_cat_test.argmax(axis=-1), y_pred.argmax(axis=-1))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

disp.plot(cmap='Blues')  # cmap define a cor
plt.show()

## [20] BÔNUS - Repeated Cross Validation para pequenos datasets

Encapsula o processo de duas etapas usado em datasets pequenos:

1. **`run_rcv()`**: repete várias vezes um treino com `validation_split` +
   `EarlyStopping`, só para descobrir em qual época (em média) o modelo
   começa a dar overfit (menor `val_loss`).
2. **`fit_final()`**: usa essa estimativa para treinar UM modelo final
   "às cegas", sem separar validação, aproveitando 100% do `X_train`
   (o objetivo de reservar validação era só descobrir o nº de épocas,
   uma vez descoberto, não precisamos mais "gastar" dados com ela).

Todos os hiperparâmetros de ambas as etapas ficam explícitos no
construtor, para facilitar comparar variações em aula.

In [ ]:
class RCVTrainer:

    def __init__(
        self,
        model_fn, compile_fn, input_shape,
        X, y, sample_weight=None,
        monitor='val_loss',
        patience=15,
        validation_split=0.15,
        rcv_epochs=100,
        batch_size=128,
        learning_rate=1e-3,
        epoch_buffer=None,      # None -> usa (1 + validation_split)
        checkpoint_path='models/abalone_test_v1.keras',
    ):
        self.model_fn = model_fn
        self.compile_fn = compile_fn
        self.input_shape = input_shape
        self.X = X
        self.y = y
        self.sample_weight = sample_weight
        self.monitor = monitor
        self.patience = patience
        self.validation_split = validation_split
        self.rcv_epochs = rcv_epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.epoch_buffer = epoch_buffer if epoch_buffer is not None else (1 + validation_split)
        self.checkpoint_path = checkpoint_path

        self.cv_histories = []
        self.best_epochs_ = None
        self.final_history_ = None
        self.final_model_ = None

    def _build_model(self):
        model = self.model_fn(self.input_shape)
        return self.compile_fn(model, self.learning_rate)

    def _fit_once(self):
        model = self._build_model()
        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor=self.monitor,
                patience=self.patience,
                restore_best_weights=True,
            )
        ]
        history = model.fit(
            self.X, self.y,
            validation_split=self.validation_split,
            epochs=self.rcv_epochs,
            batch_size=self.batch_size,
            shuffle=True,
            sample_weight=self.sample_weight,
            callbacks=callbacks,
            verbose=0,
        )
        return pd.DataFrame(history.history)

    def run_rcv(self, n_iters=10, verbose=1):
        """Repete o treino com early stopping `n_iters` vezes, guardando o histórico de cada rodada."""
        self.cv_histories = []
        for i in range(n_iters):
            if verbose:
                print(f'ITERATION {i+1}/{n_iters}... START', end='')
            history = self._fit_once()
            self.cv_histories.append(history)
            if verbose:
                print(f' shape={history.shape} ... END.')
        return self.cv_histories

    def epoch_stats(self):
        """Média e desvio padrão da melhor época (menor `monitor`) entre as rodadas de RCV."""
        if not self.cv_histories:
            raise RuntimeError('Rode run_rcv() antes de estimar as épocas.')
        best_epochs = [h[self.monitor].argmin() for h in self.cv_histories]
        self.best_epochs_ = (np.mean(best_epochs), np.std(best_epochs))
        return self.best_epochs_

    def plot_epoch_distribution(self):
        """Dispersão da melhor época entre as rodadas: std pequeno frente à média = estimativa confiável."""
        best_epochs = [h[self.monitor].argmin() for h in self.cv_histories]
        avg, std = self.best_epochs_ if self.best_epochs_ else self.epoch_stats()

        plt.figure(figsize=(8, 4))
        plt.hist(best_epochs, bins=range(min(best_epochs), max(best_epochs) + 2))
        plt.axvline(avg, color='red', linestyle='--', label=f'média = {avg:.1f} (std = {std:.1f})')
        plt.title('Distribuição da melhor época entre as rodadas de RCV')
        plt.xlabel('Época')
        plt.ylabel('Frequência')
        plt.legend()
        plt.show()

    def estimated_final_epochs(self):
        """N de épocas do treino final = média da RCV * buffer de tolerância (`epoch_buffer`)."""
        avg, _ = self.best_epochs_ if self.best_epochs_ else self.epoch_stats()
        return round(avg * self.epoch_buffer)

    def fit_final(self, epochs=None, verbose=0):
        """Treino final às cegas (sem validação), usando 100% de X, pelo nº de épocas estimado."""
        epochs = epochs if epochs is not None else self.estimated_final_epochs()

        model = self._build_model()

        # no Colab a pasta do checkpoint não existe ainda
        os.makedirs(os.path.dirname(self.checkpoint_path) or '.', exist_ok=True)

        callbacks = [
            keras.callbacks.ModelCheckpoint(
                self.checkpoint_path,
                save_best_only=True,
                monitor='loss', # Agora não existe mais val_loss!
            )
        ]
        history = model.fit(
            self.X, self.y,
            epochs=epochs,
            batch_size=self.batch_size,
            shuffle=True,
            sample_weight=self.sample_weight,
            callbacks=callbacks,
            verbose=verbose,
        )
        self.final_model_ = model
        self.final_history_ = pd.DataFrame(history.history)
        print(f'Treino final concluído: {epochs} épocas | checkpoint: "{self.checkpoint_path}"')
        return self.final_history_

## [21] RCV: estimando o nº de épocas seguro até o overfit

Repita o treinamento com diferentes splits múltiplas vezes, quanto mais,
melhor a significância estatística.

- **objetivo 1**: tunar o modelo em um train/val significativo;
- **objetivo 2**: encontrar a quantidade de épocas mais segura até o early
  stopping, antes do treino final.

Esta célula treina 10 modelos: é a mais demorada do notebook.

In [ ]:
trainer = RCVTrainer(
    model_fn=dual_head_mlp_4,
    compile_fn=compile_model,
    input_shape=X_train_scaled.shape[1:],
    X=X_train_scaled,
    y={"reg_output": y_reg_train, "cat_output": y_cat_train},
    sample_weight=sample_weight_dict,
    monitor='val_loss',
    patience=15,
    validation_split=0.15,
    rcv_epochs=100,
    batch_size=128,
    learning_rate=1e-3,
    checkpoint_path='models/abalone_test_v1.keras',
)

cv_histories = trainer.run_rcv(n_iters=10)

print(f'{len(cv_histories)} rodadas concluídas')

## [22] RCV: estatísticas e diagnóstico da melhor época

Verifique se o desvio padrão é pequeno em relação à média: quanto menor,
menos incerta é a estimativa da melhor época.

In [ ]:
avg_epochs, std_epochs = trainer.epoch_stats()
print(f'época média: {avg_epochs:.1f} | std: {std_epochs:.1f}')

trainer.plot_epoch_distribution()

## [23] Treino final "às cegas" com o nº de épocas estimado

A média da melhor época de validação é um ponto de partida para encontrar
o melhor treinamento.

Geralmente, adiciona-se um buffer de épocas proporcional à % do split de
validação utilizado na RCV (parâmetro `epoch_buffer`, default
= 1 + `validation_split`).

O treino final para benchmark utiliza todo o `X_train` sem validação, às
cegas, pelo número de épocas estimado, e restaura o modelo com melhor loss.

In [ ]:
print(f'épocas estimadas para o treino final: {trainer.estimated_final_epochs()}')

final_history = trainer.fit_final()

In [ ]:
plot_history(final_history, skip_epochs=0, smooth=5)

Por fim, recarregamos o melhor checkpoint e medimos o desempenho no
conjunto de teste — que nunca foi visto em nenhuma etapa.

In [ ]:
best_model = keras.models.load_model(trainer.checkpoint_path)

best_model.evaluate(
    X_test_scaled,
    {'reg_output': y_reg_test, 'cat_output': y_cat_test},
    return_dict=True
)